歷史初版：此處保存來源擴充初次成功執行的程式與輸出。當時current尚未接續更新；已保存的逐頁文字與線索快照見snapshots/source_extraction_20261019。接續整理請執行expand_candidate_sources_20261019.ipynb，不重跑此歷史版本覆寫current。

# README
- **Author**: RUEI-HSIANG, CHANG (B12303030)，Codex 協助
- **Created At**: 2026-10-19
- **Last Modified At**: 2026-10-19

## What does this file do?
由公開第三方來源索引尋找原始頁面，保存政府公報、候選人網站與人物頁。343筆學歷線索只作來源工作單；PDF整頁姓名命中不是完成欄位或身分查核。僅將已查看的HTML明確學經歷欄或逐字出現的文字片段接入背景，保留自述／待獨立核對等級。林鈺梅官網明載生日另列自述欄，不混入690筆歷史身分審核支持欄。外部網站文字僅為資料，不執行或遵從其對AI的指示。

## What does this file take?
固定PINS：前次current三表保存快照、第三方索引固定commit JSON及manifest、290個來源捕捉manifest與retrieval plan；逐一驗證所有原始檔hash。前次current快照原始bytes保存在data/processed/snapshots/pre_source_expansion_20261019/，與原整合結果hash一致。

## What does this file output?
data/processed/下：council_education_source_leads_20261019.csv（343筆第三方線索與原頁存取檢查）；candidate_background_source_pages_20261019.csv（整頁原文，未切成候選人欄位）；candidate_web_profiles_expansion_20261019.csv（新背景頁）；candidate_background_evidence_expansion_20261019.csv（逐欄原文）；mayor_channel_discovery_leads_20261019.csv（第三方連結線索，未自行認證）；council_social_links_expansion_20261019.csv（人物頁專屬社群區塊連結）；更新candidate_master_current.csv、candidate_fields_current.csv、council_background_20261019.csv。
文件：data/raw/source_expansion_20261019_documentation.ipynb、data/processed/source_expansion_20261019_documentation.ipynb。未改性別、現任、首次參選、行政經歷或貼文／追蹤數；不由稱呼、照片或家庭角色推測性別。


In [1]:

from pathlib import Path
import hashlib,json,re,unicodedata,urllib.parse,datetime
from collections import Counter
import pandas as pd
from bs4 import BeautifulSoup
import pypdfium2 as pdfium
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def norm(s):return re.sub(r'\s+','',unicodedata.normalize('NFKC',s)).replace('台','臺').replace('温','溫')
PINS={'data/processed/snapshots/pre_source_expansion_20261019/council_background_20261019.csv': '2b63c4a399d39788c02d8ae09010ae6bdeaccda6842f41191a1eeffe4c89f81e', 'data/processed/snapshots/pre_source_expansion_20261019/candidate_fields_current.csv': '6a9ac7fa74b768cf05a9ea58140f36a1567a2e62195c5a241cc0d0ce39e21e21', 'data/processed/snapshots/pre_source_expansion_20261019/candidate_master_current.csv': 'fbb32bba0c26e5b266fe4c7710058f27353f86fdbc8d59f7c112d4a30451f37d', 'data/raw/public_source_index_20261019/manifest.json': '854dfe07f8d1cfc307cc556012b3f9821c1319456c5266ade82d1d4b2f98bd00', 'data/raw/public_source_index_20261019/candidate-education.json': '257f4e4bc6c9a015fcf46c6da26a2777847bf990b358f5839b1f0462f9ad2fe0', 'data/raw/public_source_index_20261019/candidate-channels.json': '13643c44720e72aced6767f05effb6d49d7576c00a55b446a5638edeec85326e', 'data/raw/background_expansion_20261019/capture_manifest.json': '12479a499c4052a55eb5f84758a661a294d60f436e45bfcf6329ac2a270f6aab', 'data/raw/background_expansion_20261019/retrieval_plan.json': 'bf3b744f2a5e43459eb2fed7382f162a6e0b12123653db1aa369d46a607d2c42'}
for p,h in PINS.items():assert sha(ROOT/p)==h,p
read=lambda p:pd.read_csv(ROOT/p,dtype=str).fillna('')
snap='data/processed/snapshots/pre_source_expansion_20261019/'
master=read(snap+'candidate_master_current.csv');fields=read(snap+'candidate_fields_current.csv');council=read(snap+'council_background_20261019.csv')
manifest=json.loads((ROOT/'data/raw/background_expansion_20261019/capture_manifest.json').read_text())
plan=json.loads((ROOT/'data/raw/background_expansion_20261019/retrieval_plan.json').read_text())
raw_index_manifest=json.loads((ROOT/'data/raw/public_source_index_20261019/manifest.json').read_text())
for r in manifest+raw_index_manifest:
 if r.get('path'):assert sha(ROOT/r['path'])==r['sha256'],r['path']
lookup={r['url']:r for r in manifest};assert len(lookup)==len(plan['urls'])==290
docs={};html={};parse_errors={}
for r in manifest:
 if r['http_status']!='200' or not r['path']:continue
 try:
  if r['content_kind']=='pdf':
   doc=pdfium.PdfDocument(str(ROOT/r['path']));pages=[]
   for i in range(len(doc)):
    page=doc[i];tp=page.get_textpage();pages.append(tp.get_text_range());tp.close();page.close()
   doc.close();docs[r['url']]=pages
  else:
   s=BeautifulSoup((ROOT/r['path']).read_bytes(),'html.parser')
   for node in s(['script','style','noscript']):node.decompose()
   html[r['url']]=s
 except Exception as err:parse_errors[r['url']]=type(err).__name__+': '+str(err)
leads=[];page_rows=[];profiles=[];evidence=[];social=[]
def add_evidence(p,field,value,rec,locator,tier,year='2026'):
 if not value:return
 value=str(value).replace('\r','').replace('\n','\\n')
 evidence.append(dict(candidate_key=p['candidate_key'],candidate_name=p['candidate_name'],county_city=p['county_city'],office_type=p['office_type'],field=field,value_raw=value,source_url=rec['url'],source_file=rec['path'],source_sha256=rec['sha256'],source_locator=locator,source_year=year,evidence_tier=tier,identity_status='source_name_and_candidate_context_reviewed_pending_independent_identity_check',checked_date='2026-10-19'))
manual={
 '陳彥廷':(['台大光電碩士'],['家樂福 15 年資深經歷']),
 '李明燦':(['清大電機碩士','成大電機學士','嘉義高中／水上國中／大同國小'],['台灣、美國與中國 3 地 25 年半導體電子業研發與品保主管','曾任聯電集團處長','唐恩書店創辦人','台灣模型車博物館 TMCM 創辦人']),
 '錢康明':(['國立台東大學兒童文學碩士','國立藝術學院（臺北藝術大學）戲劇系 藝術學學士'],['前新竹市文化局局長','台中國家歌劇院評鑑委員暨評議委員']),
 '林妤芬':(['美國加州大學柏克萊分校','法學碩士（LL.M.）','輔仁大學財經法律系','法學士'],['新竹市政府廉政會報委員','新竹縣警局諮詢委員','弘準國際法律事務所','主持律師']),
 '陳又新':(['台大法律系及社工系雙學位學士','交大科法所碩士'],[]),
 '林彥':(['國立臺灣科技大學 工學院學士','國立高雄應用科技大學 商學院學士','國立高雄科技大學 商學院碩士'],['曾於 高雄市政府 負責觀光旅宿管理','也曾在 高雄榮民總醫院 推動各行政區與醫療單位的公共設施改善','並於 國立傳統藝術中心（臺灣豫劇團）負責人事與總務等公共事務']),
 '陳力新':(['國立政治大學 EMBA 碩士','台大 E 勢泮後 EMBA 結業'],['國際企業負責人']),
 '林鈺梅':(['國立中興大學企業管理學系','國立中興大學應用經濟學系碩士在職專班','順利完成碩士學位'],['日月光集團（ASE Group）旗下環隆電氣（USI）任職','科技業服務七年']),
 '許纘融':(['中國醫藥大學製藥碩士畢業','成都中醫藥大學臨床醫學畢業'],['現於南投在地擔任專業看護與按摩師傅']),
 '蕭告':(['國立臺灣大學法律學系司法組畢業','加州大學舊金山法學院（UC Hastings）交換學生'],['蕭告律師事務所主持律師']),
}
seen_profiles=set()
for lead in plan['education_leads']:
 r=lookup[lead['fetch_url']];p=master[master.candidate_key.eq(lead['candidate_key'])];assert len(p)==1;p=p.iloc[0].to_dict()
 host=urllib.parse.urlparse(r['url']).hostname or '';primary_gov=host.endswith('.gov.tw');name=lead['candidate_name'];nname=norm(name)
 requested=urllib.parse.parse_qs(urllib.parse.urlparse(lead['upstream_source_url']).fragment).get('page',[''])[0]
 pages=docs.get(r['url'],[]);hits=[i+1 for i,t in enumerate(pages) if nname in norm(t)]
 if requested and int(requested) in hits:selected=[int(requested)]
 else:selected=hits
 page_name_status='name_present_on_requested_page' if requested and int(requested) in hits else 'name_present_on_other_pages' if hits else 'name_not_found_in_text_layer' if pages else 'not_a_readable_saved_pdf'
 htext=html[r['url']].get_text(' ',strip=True) if r['url'] in html else ''
 html_name=bool(htext and nname in norm(htext))
 leads.append(dict(candidate_key=p['candidate_key'],candidate_name=name,county_city=p['county_city'],electoral_district=p['electoral_district'],upstream_claim_raw=json.dumps(lead['upstream_claim'],ensure_ascii=False),upstream_source_type=lead['upstream_source_type'],upstream_source_year=str(lead['upstream_source_year'] or ''),source_url=lead['upstream_source_url'],fetch_url=r['url'],http_status=r['http_status'],source_file=r['path'],source_sha256=r['sha256'],document_is_government=str(primary_gov),requested_page=requested,actual_name_pages=' | '.join(map(str,hits)),page_name_status=page_name_status,html_name_present=str(html_name),validation_status='source_saved_name_found_fields_and_identity_pending' if hits or html_name else 'source_access_or_name_check_pending',parse_error=parse_errors.get(r['url'],''),upstream_repository='weryk153/legislator-background',upstream_commit=raw_index_manifest[0]['commit']))
 for no in selected:
  page_rows.append(dict(candidate_key=p['candidate_key'],candidate_name=name,county_city=p['county_city'],source_url=lead['upstream_source_url'],fetch_url=r['url'],source_file=r['path'],source_sha256=r['sha256'],page=no,page_text=pages[no-1].replace('\r','').replace('\n','\\n'),page_role='whole_source_page_not_segmented_candidate_fields',identity_status='historical_name_match_pending_independent_linkage',government_source=str(primary_gov)))
 if lead['upstream_source_type']!='website' or not html_name or p['candidate_key'] in seen_profiles:continue
 s=html[r['url']];ed=[];ex=[];tier='candidate_website_self_declaration_not_degree_certification';year='2026';loc='manually selected literal source spans'
 if 'pridewatch.tw' in host:
  ednode=s.select_one('.views-field-field-education .field-content');exnode=s.select_one('.views-field-field-qualifications .field-content')
  ed=[ednode.get_text(' | ',strip=True)] if ednode else [];ex=[exnode.get_text(' | ',strip=True)] if exnode else []
  tier='civil_society_candidate_profile_declaration_not_independent_certification';loc='.views-field-field-education/.views-field-field-qualifications .field-content'
  current=s.select_one('.views-field-field-election-position-2026 .field-content');year='2026' if current and current.get_text(strip=True) else '2022_or_earlier_profile_not_currently_reconfirmed'
  for a in s.select('ul.candidate-social-links a[href]'):
   social.append(dict(candidate_key=p['candidate_key'],candidate_name=name,county_city=p['county_city'],account_url=a['href'],source_url=r['url'],source_file=r['path'],status='candidate_specific_social_block_link_target_content_pending'))
 elif host=='newpowerparty.tw':
  for heading in s.select('h5'):
   label=heading.get_text(strip=True)
   if label not in ['學歷','經歷']:continue
   container=heading.find_parent('div',class_='e-con');node=container.select_one('.elementor-widget-text-editor') if container else None
   if node:(ed if label=='學歷' else ex).append(node.get_text(' | ',strip=True))
  tier='official_party_candidate_profile_self_declaration';loc='h5 學歷／經歷所在e-con之text-editor'
 elif name=='應佳妤':
  for block in s.select('[data-cy="profile-info-content"]'):
   if '【學歷】' not in block.get_text():continue
   mode=''
   for node in block.find_all('p'):
    t=node.get_text(' ',strip=True)
    if t in ['【學歷】','【經歷】']:mode=t;continue
    if t.startswith('【'):mode=''
    if t and mode:(ed if mode=='【學歷】' else ex).append(t)
  loc='profile-info-content中【學歷】／【經歷】段落'
 elif name in manual:
  ed,ex=manual[name]
  for value in ed+ex:assert norm(value) in norm(htext),(name,value)
 # The profile URL is useful even when no defensible field extraction exists.
 add_evidence(p,'profile_url',r['url'],r,'candidate/person background page with matching name',tier,year)
 if ed:add_evidence(p,'education',' | '.join(ed),r,loc,tier,year)
 if ex:add_evidence(p,'experience',' | '.join(ex),r,loc,tier,year)
 profiles.append(dict(candidate_key=p['candidate_key'],candidate_name=name,county_city=p['county_city'],electoral_district=p['electoral_district'],profile_url=r['url'],education_raw=' | '.join(ed),experience_raw=' | '.join(ex),source_file=r['path'],source_sha256=r['sha256'],source_year=year,evidence_tier=tier,identity_status='name_and_candidate_context_lead_pending_independent_verification'))
 seen_profiles.add(p['candidate_key'])
 if name=='林鈺梅':
  assert norm('民國81年10月17日鈺梅出生於臺中縣') in norm(htext)
  add_evidence(p,'birth_date','1992-10-17',r,'出生・成長：民國81年10月17日鈺梅出生於臺中縣','candidate_website_explicit_birth_date_self_declaration','2026')
leads=pd.DataFrame(leads);pages=pd.DataFrame(page_rows).drop_duplicates(['candidate_key','fetch_url','page']);profiles=pd.DataFrame(profiles);evidence=pd.DataFrame(evidence)
assert len(leads)==343 and leads.candidate_key.nunique()==341
assert profiles.candidate_key.is_unique
assert not evidence.duplicated(['candidate_key','field','source_url']).any()
def update(key,field,value,url,status,note):
 mask=fields.candidate_key.eq(key)&fields.field.eq(field);assert mask.sum()==1
 fields.loc[mask,['existing_value','evidence','status','note','next_source','updated_date']]=[value,url,status,note,'官方公報／獨立身分及欄位複核','2026-10-19']
 mask=master.candidate_key.eq(key);assert mask.sum()==1;master.loc[mask,field+'_status']=status
for r in evidence.to_dict('records'):
 if r['field'] in ['profile_url','education','birth_date']:
  field='profile' if r['field']=='profile_url' else r['field']
  update(r['candidate_key'],field,r['value_raw'],r['source_url'],'背景原文已保存，來源自述／人物對應待獨立複核',r['source_locator']+'；來源年：'+r['source_year']+'；非戶籍或學位認證。')
# Keep history-supported fields unchanged and separate new self-declared observations.
council['additional_profile_url']='';council['additional_profile_status']='';council['birth_date_self_declared']='';council['birth_date_self_declared_source']=''
for r in profiles.to_dict('records'):
 mask=council.candidate_key.eq(r['candidate_key']);assert mask.sum()==1
 council.loc[mask,['additional_profile_url','additional_profile_status']]=[r['profile_url'],r['identity_status']]
 if r['education_raw']:council.loc[mask,'education_raw']=r['education_raw']
 if r['experience_raw']:council.loc[mask,'experience_raw']=r['experience_raw']
 council.loc[mask,'background_source_url']=r['profile_url']
for r in evidence[evidence.field.eq('birth_date')].to_dict('records'):
 mask=council.candidate_key.eq(r['candidate_key']);council.loc[mask,['birth_date_self_declared','birth_date_self_declared_source']]=[r['value_raw'],r['source_url']]
# Preserve all mayor links as third-party leads, without upgrading to official verification.
channel=json.loads((ROOT/'data/raw/public_source_index_20261019/candidate-channels.json').read_text());mayor_leads=[]
for p in channel['candidates']:
 county=p['role'].replace('長','').replace('市','市')
 matches=master[master.candidate_name.map(norm).eq(norm(p['name'])) & master.office_type.str.contains('長')]
 if len(matches)!=1:continue
 person=matches.iloc[0]
 for link in p['links']:
  mayor_leads.append(dict(candidate_key=person.candidate_key,candidate_name=person.candidate_name,county_city=person.county_city,kind=link['kind'],url=link['url'],upstream_evidence_url=link.get('evidence',{}).get('url',''),upstream_note=link.get('evidence',{}).get('note',''),verification_status='third_party_discovery_lead_not_independently_reverified',upstream_repository='weryk153/legislator-background',upstream_commit=raw_index_manifest[0]['commit']))
mayor_leads=pd.DataFrame(mayor_leads)
social=pd.DataFrame(social,columns=['candidate_key','candidate_name','county_city','account_url','source_url','source_file','status']).drop_duplicates()
outputs={'data/processed/snapshots/source_extraction_20261019/council_education_source_leads_20261019.csv':leads,'data/processed/snapshots/source_extraction_20261019/candidate_background_source_pages_20261019.csv':pages,'data/processed/candidate_web_profiles_expansion_20261019.csv':profiles,'data/processed/candidate_background_evidence_expansion_20261019.csv':evidence,'data/processed/mayor_channel_discovery_leads_20261019.csv':mayor_leads,'data/processed/council_social_links_expansion_20261019.csv':social,'data/processed/candidate_master_current.csv':master,'data/processed/candidate_fields_current.csv':fields,'data/processed/council_background_20261019.csv':council}
verification={}
for path,frame in outputs.items():
 assert path not in PINS and (ROOT/path).is_relative_to(ROOT/'data/processed')
 frame.to_csv(ROOT/path,index=False,encoding='utf-8-sig');d=read(path)
 assert len(d)==len(frame) and list(d.columns)==list(frame.columns)
 assert len((ROOT/path).read_text(encoding='utf-8-sig').splitlines())==len(d)+1,path
 verification[path]={'rows':len(d),'sha256':sha(ROOT/path)}
assert len(master)==1583 and master.candidate_key.is_unique and len(fields)==31660 and not fields.duplicated(['candidate_key','field']).any()
assert len(council)==1502 and council.birth_date_supported_history.ne('').sum()==690 and council.sex_supported_history.ne('').sum()==690
before=read(snap+'candidate_fields_current.csv').set_index(['candidate_key','field']);after=fields.set_index(['candidate_key','field'])
for field in ['sex','incumbency','first_time_this_office','first_time_any_office','administrative_experience','posts_Facebook','posts_Instagram','posts_Threads','followers']:
 pd.testing.assert_frame_equal(before.xs(field,level='field'),after.xs(field,level='field'))
summary={'retrieved_urls':len(manifest),'http_200':sum(r['http_status']=='200' for r in manifest),'saved_pdf_documents':len(docs),'education_lead_people':341,'education_leads':343,'source_name_found_people':int(leads[leads.validation_status.eq('source_saved_name_found_fields_and_identity_pending')].candidate_key.nunique()),'page_records':len(pages),'new_background_profile_people':len(profiles),'new_education_people':int(profiles.education_raw.ne('').sum()),'new_experience_people':int(profiles.experience_raw.ne('').sum()),'no_profile_after_expansion':int((council.moi_profile_url.eq('') & council.campaign_profile_url.eq('') & council.additional_profile_url.eq('')).sum()),'mayors_with_discovery_links':int(mayor_leads.candidate_key.nunique()),'mayor_link_records':len(mayor_leads),'source_specific_social_links':len(social),'parse_errors':parse_errors,'outputs':verification}
(ROOT/'data/code/source_expansion_20261019_verification.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))
print(json.dumps(summary,ensure_ascii=False,indent=2))


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


{
  "retrieved_urls": 290,
  "http_200": 265,
  "saved_pdf_documents": 171,
  "education_lead_people": 341,
  "education_leads": 343,
  "source_name_found_people": 252,
  "page_records": 171,
  "new_background_profile_people": 22,
  "new_education_people": 21,
  "new_experience_people": 20,
  "no_profile_after_expansion": 763,
  "mayors_with_discovery_links": 62,
  "mayor_link_records": 85,
  "source_specific_social_links": 9,
  "parse_errors": {
    "https://eebulletin.cec.gov.tw/111/14%E5%B1%8F%E6%9D%B1%E7%B8%A3/02%E7%B8%A3%E8%AD%B0%E5%93%A1/%E7%AC%AC1%E9%81%B8%E5%8D%801.pdf": "PdfiumError: Failed to load document (PDFium: Data format error).",
    "https://eebulletin.cec.gov.tw/111/03%E6%96%B0%E5%8C%97%E5%B8%82/02%E5%B8%82%E8%AD%B0%E5%93%A1/%E5%8D%80%E5%9F%9F%E8%AD%B0%E5%93%A1%E9%81%B8%E8%88%89%E5%85%AC%E5%A0%B1%E7%AC%AC6%E9%81%B8%E8%88%89%E5%8D%80.pdf": "PdfiumError: Failed to load document (PDFium: Data format error).",
    "https://eebulletin.cec.gov.tw/111/11%E5%8D%97%E6%8A%95%E

2026-10-20 維護：重複檔案改用單一保存路徑，固定指紋同步更新；議員登記CSV排版換行改為空格，政黨標籤不變。本次已獨立從raw表格重建並比對全部1,502筆及16欄；未重跑本Notebook或整條研究流程，既有執行輸出屬先前執行紀錄。